# 04. Ampliación de familias de modelos con MLflow

**Proyecto**: Pronóstico de demanda multi-series (Curso II — Especialización ML Engineering)

Continúa la **Fase 3** del notebook `03_entrenamiento_mlflow.ipynb` (baselines + LightGBM).
Aquí se prueban las **familias restantes** del plan con la estrategia global (un solo modelo
entrena sobre las features de las 150 series):

- **GBM**: XGBoost, CatBoost.
- **Ensambles**: RandomForest, ExtraTrees.
- **Redes**: MLPRegressor.

Cada modelo se entrena, se evalúa sobre el holdout (oct–dic 2017) y se registra en MLflow
como run + Model Registry, igual que en el notebook 03. Al final se promueve el mejor según
la regla de decisión (ranking **MASE + WAPE**, filtrando `|rel_bias| <= 5%`).

Las configuraciones viven en un **solo lugar**: `src/models/configs.py` (lo usan también
`scripts/train.py` y este notebook), para que la comparación sea con los mismos parámetros.

In [1]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from mlflow.tracking import MlflowClient

# Localiza la raíz del proyecto estés donde estés (desde notebooks/ o desde la raíz)
def _find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw" / "train.csv").exists():
            return p
    raise FileNotFoundError("No se encontró la raíz del proyecto (data/raw/train.csv)")

ROOT = _find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

from src.features.build_features import FEATURE_COLUMNS  # noqa: E402
from src.models.metrics import naive_scale_by_series  # noqa: E402
from src.models.train_model import (  # noqa: E402
    MAX_REL_BIAS_PCT,
    promote_best_model,
    train_and_log_model,
)
from src.models.configs import all_configs  # noqa: E402

sns.set_theme()
pd.set_option("display.float_format", "{:.2f}".format)

DATA_OUT = ROOT / "data" / "processed"
TRACKING_URI = f"sqlite:///{(ROOT / 'mlruns' / 'mlflow.db').as_posix()}"
EXPERIMENT_NAME = "demand_forecast_fase3"
REGISTERED_MODEL = "demand_forecast"
TARGET = "sales"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print("OK")


E:\JAIME\2023 3000\Estadística\18 Gemini CLI\ML2_Series_de_tiempo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK


## 1. Carga de datos

Se cargan las features ya guardadas, la escala naive por serie (denominador del MASE)
y el `df_test` para cruzar predicciones con lo real.

In [2]:
def load_data():
    train = pd.read_csv(DATA_OUT / "train_features.csv")
    holdout = pd.read_csv(DATA_OUT / "holdout_features.csv")
    train = train.dropna().reset_index(drop=True)
    holdout = holdout.dropna().reset_index(drop=True)

    features = [c for c in FEATURE_COLUMNS if c in train.columns]
    X_train, y_train = train[features], train[TARGET]
    X_test, y_test = holdout[features], holdout[TARGET]

    df_test = holdout[["date", "store", "item", "sales"]].reset_index(drop=True)
    series_ids_test = pd.Series(list(zip(df_test["store"], df_test["item"])))

    train_raw = pd.read_csv(DATA_OUT / "train.csv")
    naive_scale = naive_scale_by_series(train_raw)
    return X_train, X_test, y_train, y_test, naive_scale, df_test, series_ids_test

X_train, X_test, y_train, y_test, naive_scale, df_test, series_ids_test = load_data()
print(f"Train: {X_train.shape} | Holdout: {X_test.shape}")
print(f"Series en holdout: {series_ids_test.nunique()}")


Train: (255600, 15) | Holdout: (13800, 15)
Series en holdout: 150


## 2. Modelos a probar (desde el catálogo central)

El catálogo vive en `src/models/configs.py` (`all_configs()`). Los baselines y LightGBM ya
quedaron registrados en el notebook 03; aquí se corren las **5 familias restantes** para
completar la comparación (no se repite lo ya entrenado).

In [3]:
all = all_configs()
NEW_MODELS = {
    name: cfg for name, cfg in all.items()
    if name in {"xgboost", "catboost", "random_forest", "extra_trees", "mlp"}
}

print("Modelos nuevos a entrenar:")
for name, cfg in NEW_MODELS.items():
    print(f"  - {name:14s} familia={cfg.family}")
print(f"\nFiltro de sesgo: |rel_bias| <= {MAX_REL_BIAS_PCT}% para Production")


Modelos nuevos a entrenar:


  - xgboost        familia=gbm
  - catboost       familia=gbm
  - random_forest  familia=ensemble
  - extra_trees    familia=ensemble
  - mlp            familia=neural_net

Filtro de sesgo: |rel_bias| <= 5.0% para Production


## 3. Entrenamiento y registro en MLflow

Cada modelo se entrena con `train_and_log_model` (parámetros + todas las métricas +
artefactos `predictions.csv` y gráfica + Model Registry). El alias `Production` se
recalcula después de cada run considerando **todos** los runs del experimento (incluye
los del notebook 03).

In [4]:
results = {}
for name, cfg in NEW_MODELS.items():
    results[name] = train_and_log_model(
        cfg,
        X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
        df_test=df_test, series_ids_test=series_ids_test, naive_scale=naive_scale,
        experiment_name=EXPERIMENT_NAME,
        registered_model_name=REGISTERED_MODEL,
        feature_names=list(X_train.columns),
    )


Registered model 'demand_forecast' already exists. Creating a new version of this model...
2026/08/09 20:33:50 WARNING mlflow.tracking._model_registry.fluent: Run with id 8a7b9d5917a74eaaa1c5448120bf692e has no artifacts at artifact path 'model', registering model based on models:/m-15d5d7a20d1149798eaac52e123f73a7 instead


Created version '10' of model 'demand_forecast'.


  -> Production actualizado: lightgbm (v9) MASE=0.581 WAPE=10.41% rel_bias=-0.25%
[xgboost] MASE=0.581 | WAPE=10.44% | bias=-0.11 | rel_bias=-0.18% | run=8a7b9d59 | v10


Registered model 'demand_forecast' already exists. Creating a new version of this model...
2026/08/09 20:34:03 WARNING mlflow.tracking._model_registry.fluent: Run with id 28599d35b01d406caa04e751f33eb867 has no artifacts at artifact path 'model', registering model based on models:/m-538ad6df46154813a6980a79ce7ed5d3 instead


Created version '11' of model 'demand_forecast'.


...

Created version '13' of model 'demand_forecast'.


  -> Production actualizado: lightgbm (v9) MASE=0.581 WAPE=10.41% rel_bias=-0.25%
[extra_trees] MASE=0.601 | WAPE=10.83% | bias=-0.12 | rel_bias=-0.21% | run=cedb1d6c | v13


Registered model 'demand_forecast' already exists. Creating a new version of this model...
2026/08/09 20:37:28 WARNING mlflow.tracking._model_registry.fluent: Run with id 3ca414a1bb064df89e8ea29240dbf59c has no artifacts at artifact path 'model', registering model based on models:/m-b46a8b7dbc0d4d8d83cd3c97d6f0b3db instead


  -> Production actualizado: lightgbm (v9) MASE=0.581 WAPE=10.41% rel_bias=-0.25%
[mlp] MASE=0.604 | WAPE=10.94% | bias=+0.39 | rel_bias=+0.66% | run=3ca414a1 | v14


Created version '14' of model 'demand_forecast'.


In [5]:
promote_best_model(EXPERIMENT_NAME, REGISTERED_MODEL, alias="Production")


  -> Production actualizado: lightgbm (v9) MASE=0.581 WAPE=10.41% rel_bias=-0.25%


True

## 4. Comparación final (todos los runs del experimento)

Se arma la tabla desde MLflow con los runs de los notebooks 03 y 04. La columna
`Production` marca la versión ganadora según la regla de decisión.

In [6]:
def comparison_table():
    client = MlflowClient()
    exp = client.get_experiment_by_name(EXPERIMENT_NAME)
    rows = []
    for r in client.search_runs([exp.experiment_id]):
        m, p, t = r.data.metrics, r.data.params, r.data.tags
        version = p.get("registered_model_version")
        if version is None or "mase" not in m or "wape" not in m:
            continue
        try:
            mv = client.get_model_version(REGISTERED_MODEL, int(version))
        except Exception:
            continue
        rows.append({
            "modelo": t.get("mlflow.runName", r.info.run_id[:8]),
            "version": int(version),
            "MASE": m["mase"], "WAPE": m["wape"],
            "bias": m.get("bias", float("nan")),
            "rel_bias%": m.get("rel_bias_pct", float("nan")),
            "Production": "Production" in (mv.aliases or []),
        })
    return pd.DataFrame(rows).sort_values("MASE").reset_index(drop=True)

comparison_table().round(3)


,modelo,version,MASE,WAPE,bias,rel_bias%,Production
0,lightgbm,9,0.58,10.41,-0.15,-0.25,True
1,lightgbm,5,0.58,10.41,-0.15,-0.25,False
2,xgboost,10,0.58,10.44,-0.11,-0.18,False
3,catboost,11,0.59,10.53,-0.23,-0.39,False
4,extra_trees,13,0.60,10.83,-0.12,-0.21,False
5,mlp,14,0.60,10.94,0.39,0.66,False
6,random_forest,12,0.61,11.03,-0.04,-0.07,False
7,seasonal_naive,7,0.88,16.01,1.44,2.44,False
8,seasonal_naive,3,0.88,16.01,1.44,2.44,False
9,mean,8,0.92,17.33,3.13,5.32,False


## Conclusiones

- Se completó la comparación de familias de la Fase 3: baselines, GBM (LightGBM, XGBoost,
  CatBoost), ensambles (RF, ExtraTrees) y red (MLP), todos con la **estrategia global**.
- El catálogo central `src/models/configs.py` garantiza que `scripts/train.py` y los
  notebooks usen los mismos parámetros (evidencia de código reusable).
- El alias `Production` apunta a la mejor versión según la regla **MASE + WAPE con filtro
  de sesgo <= 5%**; se consume con `models:/demand_forecast@Production`.
- Pendiente de Fase 3: backtesting **walk-forward** (métricas online) en el notebook 05.
